# 01 — Exploratory Data Analysis
## Amazon Reviews 2023 — Electronics Category

This notebook explores dataset characteristics before training any models:
- Review text length distribution
- Product document length distribution  
- Reviews per product (long-tail)
- **Vocabulary mismatch analysis** — the key motivation for dense retrieval
- Split statistics (no product leakage)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
Path('../results').mkdir(parents=True, exist_ok=True)
print('Libraries loaded.')

In [ ]:
# Load dataset
DATA_DIR = '../data/'

train_df  = pd.read_parquet(f'{DATA_DIR}/train.parquet')
val_df    = pd.read_parquet(f'{DATA_DIR}/val.parquet')
test_df   = pd.read_parquet(f'{DATA_DIR}/test.parquet')
corpus_df = pd.read_parquet(f'{DATA_DIR}/corpus.parquet')

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f'Train:  {len(train_df):,} reviews ({100*len(train_df)/len(all_df):.0f}%)')
print(f'Val:    {len(val_df):,} reviews ({100*len(val_df)/len(all_df):.0f}%)')
print(f'Test:   {len(test_df):,} reviews ({100*len(test_df)/len(all_df):.0f}%)')
print(f'Corpus: {len(corpus_df):,} unique products')
print(f'Total:  {len(all_df):,} review-product pairs')

In [ ]:
# Review and product doc lengths
all_df['review_len'] = all_df['review_text'].str.split().str.len()
corpus_df['doc_len'] = corpus_df['product_doc'].str.split().str.len()

print('Review text statistics (words):')
print(all_df['review_len'].describe().round(1))
print('\nProduct doc statistics (words):')
print(corpus_df['doc_len'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Review length histogram
axes[0].hist(all_df['review_len'].clip(0, 200), bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(all_df['review_len'].median(), color='red', linestyle='--', lw=2,
                label=f'Median = {all_df["review_len"].median():.0f} words')
axes[0].set_xlabel('Review Length (words)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Review Text Length Distribution', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

# Product doc length histogram
axes[1].hist(corpus_df['doc_len'].clip(0, 300), bins=50, color='darkorange', alpha=0.8, edgecolor='white')
axes[1].axvline(corpus_df['doc_len'].median(), color='red', linestyle='--', lw=2,
                label=f'Median = {corpus_df["doc_len"].median():.0f} words')
axes[1].set_xlabel('Product Doc Length (words)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Product Document Length Distribution', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('../results/eda_lengths.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/eda_lengths.png')

In [ ]:
# Reviews per product — long-tail distribution
reviews_per_product = all_df['product_id'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bins = [1, 2, 3, 5, 10, 20, 50, 100, reviews_per_product.max()+1]
ax.hist(reviews_per_product.values, bins=50, color='purple', alpha=0.7, edgecolor='white')
ax.set_xlabel('Reviews per Product', fontsize=12)
ax.set_ylabel('Number of Products (log scale)', fontsize=12)
ax.set_title('Reviews per Product — Long-Tail Distribution', fontsize=14, fontweight='bold')
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('../results/eda_reviews_per_product.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Products with 1 review:    {(reviews_per_product==1).sum():,} ({100*(reviews_per_product==1).mean():.1f}%)')
print(f'Products with 2-5 reviews: {((reviews_per_product>=2)&(reviews_per_product<=5)).sum():,}')
print(f'Products with >5 reviews:  {(reviews_per_product>5).sum():,}')
print(f'Max reviews for 1 product: {reviews_per_product.max()}')

In [ ]:
# Vocabulary mismatch analysis — KEY motivation for dense retrieval
# BM25 depends on shared vocabulary; dense retrieval bridges this gap

stop = {'the','a','an','is','it','was','and','or','for','to','with','this','that',
        'my','i','in','on','of','but','so','very','really','great','good','nice'}

def tokenize(text):
    return set(re.findall(r'\w+', text.lower()))

# Merge reviews with product docs
sample = all_df.merge(corpus_df[['product_id','product_doc']], on='product_id', how='inner').sample(1000, random_state=42)

overlaps = []
for _, row in sample.iterrows():
    q_tok = tokenize(row['review_text']) - stop
    d_tok = tokenize(row['product_doc']) - stop
    if q_tok:
        overlap = len(q_tok & d_tok) / len(q_tok)
        overlaps.append(overlap)

overlaps = np.array(overlaps)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(overlaps, bins=50, color='crimson', alpha=0.8, edgecolor='white')
ax.axvline(np.median(overlaps), color='navy', linestyle='--', lw=2,
           label=f'Median overlap = {np.median(overlaps):.1%}')
ax.set_xlabel('Query-Document Word Overlap (Jaccard proxy)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Vocabulary Mismatch Distribution\n(BM25 fails at 0% overlap → dense retrieval shines)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../results/eda_vocab_overlap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean word overlap:  {overlaps.mean():.1%}')
print(f'Median overlap:     {np.median(overlaps):.1%}')
print(f'Zero overlap:       {(overlaps==0).sum()} pairs ({100*(overlaps==0).mean():.1f}%)')
print(f'<5% overlap:        {(overlaps<0.05).sum()} pairs ({100*(overlaps<0.05).mean():.1f}%)')
print()
print('→ These low-overlap pairs are exactly where BM25 fails and dense retrieval wins!')

In [ ]:
# Motivating examples
print('=' * 70)
print('MOTIVATING EXAMPLES — Vocabulary Mismatch')
print('(BM25 score ≈ 0; dense retrieval can still find the right product)')
print('=' * 70)

shown = 0
for _, row in sample.iterrows():
    q_tok = tokenize(row['review_text']) - stop
    d_tok = tokenize(row['product_doc']) - stop
    if not q_tok:
        continue
    overlap = len(q_tok & d_tok) / len(q_tok)
    if overlap < 0.05 and len(row['review_text'].split()) > 8:
        print(f'\n  Word overlap: {overlap:.0%}')
        print(f'  Review:  "{row["review_text"][:130]}"')
        print(f'  Product: "{row["product_doc"][:130]}"')
        shown += 1
    if shown >= 5:
        break

In [ ]:
# Split statistics
train_products = set(train_df['product_id'])
val_products   = set(val_df['product_id'])
test_products  = set(test_df['product_id'])

print('=== SPLIT STATISTICS (product-level split) ===')
print(f'Train products: {len(train_products):,}')
print(f'Val products:   {len(val_products):,}')
print(f'Test products:  {len(test_products):,}')
print(f'Corpus:         {len(corpus_df):,} unique products')
print()
print(f'Train ∩ Test overlap: {len(train_products & test_products)} (must be 0)')
print(f'Train ∩ Val overlap:  {len(train_products & val_products)} (must be 0)')
print(f'Val ∩ Test overlap:   {len(val_products & test_products)} (must be 0)')
print()
print('→ Product-level split: model never sees test products during training')
print('→ This is a realistic retrieval setting (new products at inference)')